In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "07-application-agent-framework/long-running-durable/lra/lra-gcp/notebooks/worked")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 02 · Human-in-the-loop: suspend for days, resume from anywhere

A `Wait` outcome parks the run: no task, no lease, no compute. An external event with the matching key resumes it. Timeouts are absolute timestamps enforced by the reaper.

In [1]:
from datetime import timedelta
import json
from lra import Engine, Event, Budget, Workflow, Next, Done, Wait, StepFailed, RunStatus, StepTask, SimulatedCrash
from lra.adapters.memory import FakeClock, FakeLLM, InMemoryStateStore, InMemoryTaskQueue, InMemoryEventBus, LocalRunner
from lra.examples import ALL_WORKFLOWS
from lra.examples.scripted import research_routes

def harness(routes=None, fail_times=0, chaos=None, lease_ttl_s=60):
    """One engine over in-memory adapters, driven the way Cloud Tasks would drive it."""
    clock, store, bus = FakeClock(), InMemoryStateStore(), InMemoryEventBus()
    queue = InMemoryTaskQueue(clock)
    llm = FakeLLM(routes=routes or research_routes(), fail_times=fail_times)
    engine = Engine(store=store, queue=queue, bus=bus, llm=llm, clock=clock, workflows=ALL_WORKFLOWS,
                    worker_id="worker-A", lease_ttl=timedelta(seconds=lease_ttl_s), chaos=chaos)
    return engine, LocalRunner(engine, queue, clock), clock, store, queue, bus

def show(run):
    print(f"{run.run_id} {run.status.value:12s} step={run.current_step} attempts={run.attempts} "
          f"wait={run.wait.key if run.wait else None} steps_used={run.budget.steps_used}")

## An approval gate

`patterns.hitl.request_approval` records what is being approved and returns `Wait`. `approval_decision` reads the event after resume and fails the run on reject/timeout.

In [2]:
from lra.patterns.hitl import request_approval, approval_decision

wf = Workflow("expense", version="1")

@wf.step(start=True)
def draft(ctx):
    ctx.state["amount"] = ctx.input["amount"]
    return Next("gate")

@wf.step()
def gate(ctx):
    if ctx.state["amount"] < 100:
        return Next("pay")                       # small amounts skip the human
    return request_approval(ctx, then="pay", gate="manager",
                            summary=f"expense of ${ctx.state['amount']}",
                            timeout=timedelta(days=3), on_timeout="fail")

PAID = []
@wf.step()
def pay(ctx):
    if ctx.state["amount"] >= 100:
        decision = approval_decision(ctx, gate="manager")   # raises StepFailed on reject / timeout
        ctx.state["approved_by"] = decision["by"]
    ctx.effect("pay", lambda: PAID.append(ctx.run_id) or {"tx": "t-1"})
    return Done({"paid": ctx.state["amount"], "by": ctx.state.get("approved_by")})

engine, runner, clock, store, queue, bus = harness()
engine.registry.register(wf)

## Suspend

In [3]:
run = engine.start("expense", {"amount": 900})
runner.run_until_idle()
r = store.get(run.run_id); show(r)
print("wait:", r.wait.model_dump(mode="json"))
print("queue:", len(queue), "| lease:", r.lease, "| approval requested event:", bus.of_type("approval.requested")[0]["summary"])
assert r.status == RunStatus.WAITING and r.wait.key == f"manager:{run.run_id}"

run_4f3b98dbb31d WAITING      step=pay attempts={'draft': 1, 'gate': 1} wait=manager:run_4f3b98dbb31d steps_used=2
wait: {'kind': 'approval', 'key': 'manager:run_4f3b98dbb31d', 'then': 'pay', 'timeout_at': '2026-01-04T00:00:00Z', 'on_timeout': 'fail'}
queue: 0 | lease: None | approval requested event: expense of $900


## Resume — idempotent and key-scoped

The API endpoint `POST /runs/{id}/events` does exactly this. Duplicates and wrong-gate events return `None`, never an error.

In [4]:
wrong = Event(run_id=run.run_id, key="cfo:" + run.run_id, payload={"decision": "approve"})
print("wrong gate:", engine.resume(wrong))
evt = Event(run_id=run.run_id, key=r.wait.key, payload={"decision": "approve", "by": "manager@example.com"})
print("first delivery:", engine.resume(evt).status.value)
print("duplicate:", engine.resume(evt))
runner.run_until_idle()
r = store.get(run.run_id); show(r); print(r.result)
assert r.status == RunStatus.SUCCEEDED and PAID == [run.run_id]

wrong gate: None
first delivery: RUNNING
duplicate: None
run_4f3b98dbb31d SUCCEEDED    step=None attempts={'draft': 1, 'gate': 1, 'pay': 1} wait=None steps_used=3
{'paid': 900, 'by': 'manager@example.com'}


## Rejection is a business failure: no retry, no compensation needed here

In [5]:
run2 = engine.start("expense", {"amount": 5000}); runner.run_until_idle()
engine.resume(Event(run_id=run2.run_id, key=f"manager:{run2.run_id}", payload={"decision": "reject", "by": "cfo", "comment": "no"}))
runner.run_until_idle()
r2 = store.get(run2.run_id); show(r2); print(r2.error)
assert r2.status == RunStatus.FAILED and r2.attempt_of("pay") == 1

run_5e5323924522 FAILED       step=None attempts={'draft': 1, 'gate': 1, 'pay': 1} wait=None steps_used=2
pay: approval manager rejected by cfo: no


## Timeout via the reaper

Nothing runs for three days. Then Cloud Scheduler calls `/internal/reap`.

In [6]:
run3 = engine.start("expense", {"amount": 250}); runner.run_until_idle()
print("before:", engine.reap()["waits_timed_out"])
clock.advance(days=3, seconds=1)
print("after 3 days:", engine.reap()["waits_timed_out"])
r3 = store.get(run3.run_id); print(r3.status.value, "|", r3.error)
assert r3.status == RunStatus.FAILED

before: []
after 3 days: ['run_a356ca1262cd']
FAILED | timed out waiting for manager:run_a356ca1262cd


## Auto-approve on timeout (`on_timeout="resume"`)

Only for effects you would be comfortable auto-approving. The resumed step sees `{"timed_out": True}` in the event payload.

In [7]:
soft = Workflow("soft_gate")
@soft.step(start=True)
def ask(ctx):
    return Wait(key=f"ok:{ctx.run_id}", then="finish", timeout=timedelta(hours=4), on_timeout="resume")
@soft.step()
def finish(ctx):
    payload = ctx.state["events"][f"ok:{ctx.run_id}"]["payload"]
    return Done({"auto_approved": bool(payload.get("timed_out"))})
engine.registry.register(soft)
run4 = engine.start("soft_gate"); runner.run_until_idle()
clock.advance(hours=4, seconds=1); engine.reap(); runner.run_until_idle()
print(store.get(run4.run_id).result)
assert store.get(run4.run_id).result == {"auto_approved": True}

{'auto_approved': True}


## Cancel while waiting

No task will ever run for a waiting run, so cancel finishes it immediately (and would compensate if anything compensable had completed).

In [8]:
run5 = engine.start("expense", {"amount": 400}); runner.run_until_idle()
engine.cancel(run5.run_id)
print(store.get(run5.run_id).status.value)
print("late approval:", engine.resume(Event(run_id=run5.run_id, key=f"manager:{run5.run_id}", payload={"decision": "approve"})))

CANCELLED
late approval: None


## The same gate in Cloud Workflows and ADK

* **Cloud Workflows**: `events.create_callback_endpoint` → send the URL to the reviewer → `events.await_callback(timeout=259200)`. See `workflows/research_approval.yaml`.
* **ADK 2**: a node yields an event with `long_running_tool_ids`; the webhook resumes the invocation with a `FunctionResponse`. See `examples/adk_agent_engine/`.